## A simple app


## Test your app


In [3]:
from ipywidgets import VBox, Button, FileUpload, Output, Label
from PIL import Image
from IPython.display import display
import io
import numpy as np
import torchvision
import torchvision.transforms as T
import torch

learn_inf = torch.jit.load("checkpoints/transfer_exported.pt")

def on_click_classify(change):
    
    # load the image that has been uploaded
    uploaded = btn_upload.value
    if not uploaded:
        return  # No file has been uploaded yet

    content = uploaded[-1]["content"]
    fn = io.BytesIO(
        content.tobytes() if hasattr(content, "tobytes") else content
    )
    
    img = Image.open(fn)
    img.load()
    
    # Clear the previous output
    out_pl.clear_output()
    
    # Display the image
    with out_pl:
        
        ratio = img.size[0] / img.size[1]
        c = img.copy()
        c.thumbnail([ratio * 200, 200])
        display(c)
        
    timg = T.ToTensor()(img).unsqueeze_(0)
    
    softmax = learn_inf(timg).data.cpu().numpy().squeeze()
    
    idxs = np.argsort(softmax)[::-1]
    
    for i in range(5):
        p = softmax[idxs[i]]
        
        landmark_name = learn_inf.class_names[idxs[i]]
        
        labels[i].value = f"{landmark_name} (prob: {p:.2f})"
        
btn_upload = FileUpload()

btn_run = Button(description="Classify")
btn_run.on_click(on_click_classify)

labels = []
for _ in range(5):
    labels.append(Label())

out_pl = Output()
out_pl.clear_output()

wgs = [Label("Please upload a picture of a landmark"), btn_upload, btn_run, out_pl]
wgs.extend(labels)

VBox(wgs)
    